# Week 2 Day 1: EDA on the Loan Approval Dataset

## EDA Summary
- The dataset has 614 rows and 13 columns: 5 numeric and 8 text columns; `Loan_ID` is only an identifier.
- The target `Loan_Status` is imbalanced: 422 approved (Y, 69%) and 192 rejected (N, 31%).
- Income and loan amount are heavily right-skewed (skew 6.5 and 2.7), with `ApplicantIncome` reaching 81,000 against a median of 3,812.
- Seven columns have missing values; `Credit_History` (50 rows, 8%) and `Self_Employed` (32 rows, 5%) are above the 5% threshold.
- Top correlations: `ApplicantIncome` vs `LoanAmount` (0.57), `Credit_History` vs approval (0.56), `CoapplicantIncome` vs `LoanAmount` (0.19). Applicants with credit history 1 are approved 80% of the time, against 8% for credit history 0.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidates = [Path("data/loan_approval.csv"), Path("../data/loan_approval.csv")]
csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    import os
    raise FileNotFoundError(f"CSV not found. Current folder is: {os.getcwd()}")

df = pd.read_csv(csv_path)
df.head()

## 1. Summary statistics

In [ ]:
print(df.shape)
df.describe()

## 2. Target variable distribution
Target: `Loan_Status` (Y = approved, N = rejected).

In [ ]:
counts = df["Loan_Status"].value_counts()
print(counts)
counts.plot(kind="bar", title="Loan_Status")
plt.xlabel("Loan_Status")
plt.ylabel("count")
plt.show()

## 3. Top 3 correlations
Target converted to 1 (Y) and 0 (N) so it can be correlated.

In [ ]:
data = df.copy()
data["Loan_Status_num"] = (data["Loan_Status"] == "Y").astype(int)
corr = data.select_dtypes(include="number").corr()

# keep only the upper triangle so each pair appears once
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack().sort_values(key=abs, ascending=False)
print(pairs.head(3))

## 4. Missing values

In [ ]:
missing_pct = df.isnull().mean() * 100
print(missing_pct[missing_pct > 0].round(1))
print("Columns above 5% missing:", list(missing_pct[missing_pct > 5].index))